## RSI and EWMA Strategy
Uses RSI as a momentum indicator combined with EWMA smoothing.
TA indicators built using the **TTR** package in R.

<p style="color:steelblue; font-weight:bold">Please run the last cell (RsiEwma function) first before starting backtest</p>

In [ ]:
library(TTR)      # Technical indicators (RSI, EMA, MACD)
library(dplyr)
library(ggplot2)
library(zoo)

## Trading data treatment

In [ ]:
ticker <- "MSTR"
data   <- read.csv(sprintf("../../../data/%s.5MIN.csv", ticker))
data$ewma <- EMA(data$close, n = 50)  # TTR EMA; approximate of Python's ewm(alpha=0.1)

## MACD & RSI

In [ ]:
df_ta <- data

macd_obj  <- MACD(data$close, nFast = 5, nSlow = 13, nSig = 8)
rsi_vals  <- RSI(data$close, n = 13)

df_ta$MACD   <- macd_obj[, "macd"]
df_ta$SIGNAL <- macd_obj[, "signal"]
df_ta$HIST   <- df_ta$MACD - df_ta$SIGNAL
df_ta$rsi    <- rsi_vals

In [ ]:
# Plot subset
idx <- 1000:nrow(df_ta)
par(mfrow = c(3, 1), mar = c(2,4,2,1))

plot(idx, df_ta$close[idx], type = "l", col = "black",
     main = "Close vs EWMA", ylab = "Price")
lines(idx, df_ta$ewma[idx], col = "blue")
grid()

plot(idx, df_ta$MACD[idx], type = "l", col = "blue",
     main = "MACD", ylab = "")
lines(idx, df_ta$SIGNAL[idx], col = "red")
grid()

plot(idx, df_ta$rsi[idx], type = "l", col = "purple",
     main = "RSI", ylab = "RSI")
abline(h = c(30, 70), lty = 2, col = "red")
grid()

par(mfrow = c(1, 1))

<h3 style="color:red; font-weight:bold">These are optimized values for RSI/EWMA. May not generalise to other tickers or timeframes. Use with caution.</h3>

In [ ]:
df_result <- RsiEwma(data, rsi_window = 5, rsi_upper = 72, rsi_lower = 9)

## Plot the trades

In [ ]:
temp <- df_result[1300:1500, ]
par(mfrow = c(2, 1), mar = c(2,4,2,1))

plot(seq_len(nrow(temp)), temp$close, type = "l",
     main = "Close & EWMA with Trade Signals", ylab = "Price")
lines(seq_len(nrow(temp)), temp$ewma, col = "blue")
text(seq_len(nrow(temp)), temp$close, labels = temp$Trade, cex = 0.5, srt = 90)
grid()

plot(seq_len(nrow(temp)), temp$rsi, type = "l", col = "purple",
     main = "RSI", ylab = "RSI")
abline(h = c(9, 72), lty = 2, col = "red")
grid()

par(mfrow = c(1, 1))

In [ ]:
plot(df_result$CumuPnL, type = "l",
     main = "Cumulative PnL", xlab = "Bar", ylab = "PnL ($)")
grid()

In [ ]:
cum_ret        <- cumprod(c(1, diff(df_result$CumuPnL) / head(df_result$CumuPnL, -1) + 1))
peak           <- max(cum_ret)
peak_idx       <- which(cum_ret == peak)[length(which(cum_ret == peak))]
trough         <- min(cum_ret[peak_idx:length(cum_ret)])
max_dd         <- (trough - peak) / peak

exits          <- df_result$Trade %in% c("LExit", "SExit")
total_trades   <- sum(exits)
wins           <- sum(df_result$PnL > 0)
losses         <- sum(df_result$PnL < 0)
win_pnl        <- sum(df_result$PnL[df_result$PnL > 0])
loss_pnl       <- sum(df_result$PnL[df_result$PnL < 0])

metrics <- data.frame(
    Metric = c("Total PnL","Wins","Losses","Wins PnL","Losses PnL",
               "Win Ratio","Total Trades","Average PnL","Max Drawdown",
               "Annualized Mean Return"),
    Value  = c(
        formatC(sum(df_result$PnL), format="f", digits=1, big.mark=","),
        wins, losses,
        formatC(win_pnl, format="f", digits=1, big.mark=","),
        formatC(loss_pnl, format="f", digits=1, big.mark=","),
        sprintf("%.1f%%", wins/total_trades*100),
        total_trades,
        formatC(sum(df_result$PnL)/total_trades, format="f", digits=1),
        sprintf("%.1f%%", max_dd*100),
        sprintf("%.1f%%", mean(diff(df_result$CumuPnL)/head(df_result$CumuPnL,-1),na.rm=TRUE)*4*24*252*100)
    )
)
print(metrics)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# RsiEwma function — run this cell first
# ──────────────────────────────────────────────────────────────────────────────
RsiEwma <- function(input_df, rsi_window = 5, rsi_upper = 80, rsi_lower = 20) {
    df            <- input_df
    df$rsi        <- RSI(df$close, n = rsi_window)
    df$Trade      <- ""
    df$PnL        <- 0.0
    short         <- FALSE
    long_pos      <- FALSE
    enter_price   <- 0
    shares_traded <- 0
    notional      <- 10000
    exit_signal   <- FALSE
    was_in_long   <- FALSE
    was_in_short  <- FALSE

    for (i in seq_len(nrow(df))) {
        if (i <= rsi_window) next
        row_close <- df$close[i]
        row_ewma  <- df$ewma[i]
        row_rsi   <- df$rsi[i]

        if (!short && !long_pos) {
            if (!was_in_short && row_close < row_ewma) {
                enter_price   <- row_close
                shares_traded <- round(notional / row_close, 0)
                short         <- TRUE
                df$Trade[i]   <- "S"
            } else if (!was_in_long && row_close > row_ewma) {
                enter_price   <- row_close
                shares_traded <- round(notional / row_close, 0)
                long_pos      <- TRUE
                df$Trade[i]   <- "L"
            }
            was_in_short <- FALSE
            was_in_long  <- FALSE
        } else {
            if (short) {
                pnl_pct <- shares_traded * (enter_price - row_close) / notional
                if (!is.na(pnl_pct) && pnl_pct < -0.05) {
                    df$PnL[i]  <- shares_traded * (enter_price - row_close)
                    df$Trade[i] <- "SExitR"
                    short <- FALSE; shares_traded <- 0
                    was_in_short <- TRUE; was_in_long <- FALSE
                } else {
                    df$Trade[i] <- "In-S"
                    if (row_close < row_ewma) exit_signal <- TRUE
                    if (exit_signal && !is.na(row_rsi) && row_rsi <= rsi_lower) {
                        df$PnL[i]  <- shares_traded * (enter_price - row_close)
                        df$Trade[i] <- "SExit"
                        short <- FALSE; shares_traded <- 0
                        exit_signal <- FALSE
                        was_in_short <- TRUE; was_in_long <- FALSE
                    }
                }
            } else if (long_pos) {
                pnl_pct <- shares_traded * (row_close - enter_price) / notional
                if (!is.na(pnl_pct) && pnl_pct < -0.05) {
                    df$PnL[i]  <- shares_traded * (row_close - enter_price)
                    df$Trade[i] <- "LExitR"
                    long_pos <- FALSE; shares_traded <- 0
                    was_in_long <- TRUE; was_in_short <- FALSE
                } else {
                    df$Trade[i] <- "In-L"
                    if (row_close > row_ewma) exit_signal <- TRUE
                    if (exit_signal && !is.na(row_rsi) && row_rsi >= rsi_upper) {
                        df$PnL[i]  <- shares_traded * (row_close - enter_price)
                        df$Trade[i] <- "LExit"
                        long_pos <- FALSE; shares_traded <- 0
                        exit_signal <- FALSE
                        was_in_long <- TRUE; was_in_short <- FALSE
                    }
                }
            }
        }
    }

    df$CumuPnL    <- cumsum(df$PnL)
    df$CumuPnL[1] <- 10000
    df$CumuPnL    <- cumsum(df$PnL) + 10000
    df
}

### Considerations
- TA parameters change over time — backtest with constant parameters creates look-ahead bias
- Results are not representative of live trading performance